In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
scaled_train = pd.read_csv('../data/processed/scaled_train.csv')

In [22]:
X, y = scaled_train.drop(columns=["result"]), scaled_train["result"]

In [23]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(1.2847038019451813), np.int64(1): np.float64(0.7888165038002172), np.int64(2): np.float64(1.0483405483405484)}


In [24]:
sample_weights = y.map(class_weights)

In [25]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Base models
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss"
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight=class_weights
)

svc = SVC(
    probability=True,
    random_state=42,
    class_weight=class_weights
)

# Ensemble
ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb),
        ("lr", lr),
        ("svc", svc)
    ],
    voting="soft"      # Uses probabilities
)

# Parameters to tune
param_grid = {
    "xgb__n_estimators": [50, 100],
    "xgb__learning_rate": [0.01, 0.05],
    "xgb__max_depth": [3, 5],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],

    "lr__C": [0.05, 0.1],

    "svc__C": [0.05, 0.1],
    "svc__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    ensemble,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y, sample_weight=sample_weights)

best_model = grid.best_estimator_

print("Best Parameters:")
print(grid.best_params_)

print("Best CV Accuracy:")
print(grid.best_score_)

# Predictions
y_pred = best_model.predict(X)

probs = best_model.predict_proba(X)

print(probs.shape)
print(probs[:5])

print("Train Accuracy:", accuracy_score(y, y_pred))


/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matches-prediction/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/ziadsamer/Projects/nations-matche

Best Parameters:
{'lr__C': 0.1, 'svc__C': 0.05, 'svc__gamma': 'auto', 'xgb__colsample_bytree': 1.0, 'xgb__learning_rate': 0.05, 'xgb__max_depth': 5, 'xgb__n_estimators': 50, 'xgb__subsample': 1.0}
Best CV Accuracy:
0.48368756215570174
(1453, 3)
[[0.27186733 0.52220612 0.20592656]
 [0.28539627 0.38699984 0.32760391]
 [0.179583   0.56109596 0.25932104]
 [0.40098366 0.28246795 0.31654838]
 [0.24206154 0.25958783 0.49835064]]
Train Accuracy: 0.6958017894012388


In [26]:
# import joblib
# joblib.dump(best_model, "../models/best_model.pkl")

In [27]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.55      0.59       377
           1       0.73      0.76      0.74       614
           2       0.69      0.73      0.71       462

    accuracy                           0.70      1453
   macro avg       0.69      0.68      0.68      1453
weighted avg       0.69      0.70      0.69      1453



In [28]:
import joblib
joblib.dump(best_model, '../models/xgboost.pkl')

['../models/xgboost.pkl']

# Inference